# STEP 1 — ANTsPy Parallel Registration Worker
## CN vs EMCI (ADNI)

**Open this notebook in 3 separate Colab tabs.**  
The **only** line you change between tabs is `SESSION_ID` in Cell 1:

| Tab | SESSION_ID | Subjects |
|-----|-----------|----------|
| Tab 1 | `0` | ~51 subjects |
| Tab 2 | `1` | ~51 subjects |
| Tab 3 | `2` | ~50 subjects |

Each tab writes to its own progress file and processes a non-overlapping slice — zero conflicts.  
After **all tabs finish**, run `STEP2_merge_and_classify.ipynb` once.

## ⚠️ Modified to handle subjects with no T1w scan

Not every subject in this cohort has a usable T1w structural image. This version changes the registration logic as follows:

1. **Previously-registered subjects are left untouched.** Anything already registered through the original T1w-mediated pipeline is detected and marked complete — it is never re-registered with a different method (see **CELL 11**).
2. **Subjects with no T1w are no longer dropped.** They go through a direct EPI→MNI fallback registration instead of being skipped (see **CELL 13**).
3. **Every subject's registration method is tracked** (`registration_method` column, and per-subject in the progress JSON) so you can check whether method correlates with diagnosis group (**CELL 7**) or with measured registration quality (**CELL 15**) — either would signal a confound worth reporting or controlling for.

**Why not register everyone the same way?** Direct EPI→MNI registration is generally less accurate than T1w-mediated registration — EPI has lower contrast/resolution and more distortion than a T1w structural, particularly around the hippocampus / medial temporal lobe, which matters for CN vs EMCI. Re-registering your existing, good T1w-based scans with the weaker direct method would only degrade data you already trust, with no real upside. Keeping the better method where T1w exists, using the EPI-only fallback only where it's unavoidable, and tracking the method per subject is the safer default.

4. **The cohort can now grow without breaking anything.** **CELL 5** scans Drive for subjects not yet in `subjects_df`, builds rows for them, and saves the combined result to a new `subjects_df_v2.csv` (your original `subjects_df.csv` is never overwritten). **CELL 9** now rebuilds the session partitions whenever it detects subjects that aren't covered yet, instead of silently leaving them unassigned.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — *** SET SESSION_ID BEFORE RUNNING ANYTHING ELSE ***
# ─────────────────────────────────────────────────────────────────────────

SESSION_ID     = 1    # ← Change to 0, 1, or 2 in each tab
N_SESSIONS     = 3    # Total parallel tabs — keep at 3
PARTITION_SEED = 42   # Must be identical across all tabs

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Mount Drive and install dependencies
# ─────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'antspyx',   '-q'], check=True)
subprocess.run(['pip', 'install', 'nilearn', 'nibabel', 'scikit-learn', '-q'], check=True)
print("✓ Dependencies installed")

Mounted at /content/drive
✓ Dependencies installed


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Imports and path configuration
# ─────────────────────────────────────────────────────────────────────────

import ants
import nibabel as nib
import numpy as np
import pandas as pd
import json, gc, hashlib, urllib.request
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Shared paths (same across all sessions) ───────────────────────────────────
DRIVE_ROOT       = Path('/content/drive/MyDrive')

# subjects_df.csv columns: original_id, participant_id, group,
#                          label_encoded, filepath, tr
SUBJECTS_DF_PATH = DRIVE_ROOT / 'enhanced_pipeline/subjects_df.csv'

# Root of the NIfTI tree:
#   Preprocessed_ADNI_NIFTI/
#     CN/<original_id>/T1w/*.nii.gz
#     EMCI/<original_id>/T1w/*.nii.gz
NIFTI_ROOT       = DRIVE_ROOT / 'Preprocessed_ADNI_NIFTI'

REG_ROOT         = DRIVE_ROOT / 'registered_fmri'
MNI_PATH         = REG_ROOT   / 'MNI152_T1_2mm_brain.nii.gz'

# ── Per-session paths (unique per tab → no file conflicts) ────────────────────
SESSION_CSV    = REG_ROOT / f'session_{SESSION_ID}_subjects.csv'
PROGRESS_FILE  = REG_ROOT / f'registration_progress_session_{SESSION_ID}.json'

for d in [REG_ROOT / 'CN', REG_ROOT / 'EMCI']:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Session {SESSION_ID} of {N_SESSIONS} configured")
print(f"  Progress file : {PROGRESS_FILE.name}")
print(f"  Partition CSV : {SESSION_CSV.name}")

✓ Session 1 of 3 configured
  Progress file : registration_progress_session_1.json
  Partition CSV : session_1_subjects.csv


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Load subjects_df (fast restore — no 15-min scan)
# ─────────────────────────────────────────────────────────────────────────

subjects_df = pd.read_csv(SUBJECTS_DF_PATH)

# Confirm the actual columns from this pipeline
print(f"✓ Loaded {len(subjects_df)} subjects")
print(f"  Columns : {list(subjects_df.columns)}")
print(f"  CN      : {(subjects_df['group'] == 'CN').sum()}")
print(f"  EMCI    : {(subjects_df['group'] == 'EMCI').sum()}")
print()
print(subjects_df[['original_id', 'group', 'tr']].head(3).to_string(index=False))

✓ Loaded 152 subjects
  Columns : ['original_id', 'participant_id', 'group', 'label_encoded', 'filepath', 'tr']
  CN      : 79
  EMCI    : 73

original_id group       tr
 002_S_0413    CN 3.003998
 002_S_0685    CN 6.021583
 002_S_1261    CN 3.024998


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 5 — Discover newly added subjects and build an updated subjects_df
#          (saved as a NEW file — the original subjects_df.csv is untouched)
# ──────────────────────────────────────────────────────────────────────
#
# The cohort has grown since subjects_df.csv was last built (now 245 CN +
# 246 EMCI converted subject directories on Drive). This cell:
#
#   1. Checks whether an updated subjects_df was already built by another
#      tab in this session (Drive is shared across all 3 tabs, so the file
#      a different tab wrote is already visible here). If so, just loads it
#      — no re-scanning, no manual copying between tabs needed.
#   2. Otherwise, scans Preprocessed_ADNI_NIFTI/CN and /EMCI for every
#      subject directory that currently exists, leaves every subject already
#      in subjects_df exactly as-is, and for directories NOT already in
#      subjects_df, finds their resting-state fMRI file and reads its TR to
#      build a new row in the same schema (original_id, participant_id,
#      group, label_encoded, filepath, tr).
#   3. Either way, saves/keeps the result as a NEW CSV file (subjects_df_v2.csv)
#      — the original subjects_df.csv is never overwritten — and re-points
#      the working `subjects_df` variable to it, so CELL 6 onward (T1w
#      discovery, partitioning, registration) automatically covers the full,
#      expanded cohort.
#
# Unlike T1w (always under a fixed `T1w/` subfolder), the fMRI scan is nested
# under a series-description folder whose name varies by acquisition protocol
# ('Resting_State_fMRI', 'Extended_Resting_State_fMRI', 'Axial_rsfMRI...',
# 'Axial_MB_rsfMRI...'), so discovery here searches recursively instead of
# assuming one fixed path.

# Set True to ignore the cached subjects_df_v2.csv below and force a fresh
# Drive scan in THIS tab (e.g. if you've added even more subjects since the
# cached file was built).
FORCE_RESCAN = False

GROUP_LABELS         = {'CN': 0, 'EMCI': 1}
SUBJECTS_DF_V2_PATH  = SUBJECTS_DF_PATH.parent / 'subjects_df_v2.csv'

def find_fmri_for_subject(original_id: str, group: str, nifti_root: Path):
    """
    Search for the resting-state fMRI NIfTI file for one subject under:
        <nifti_root>/<group>/<original_id>/**/*.nii.gz

    Excludes anything that looks structural (T1w/MPRAGE/FLAIR/etc.) and
    anything that isn't a 4D volume. If multiple qualifying scans are found
    (e.g. more than one session was downloaded), keeps the one with the most
    timepoints and reports the count so it can be sanity-checked manually.

    Returns (filepath: str | None, n_candidates: int)
    """
    subject_dir = nifti_root / group / original_id
    if not subject_dir.exists():
        return None, 0

    exclude_kw = ['t1w', 'mprage', 't1_', 'flair', 'localizer',
                  'field_map', 'fieldmap', 'dti', 'asl', 'calibration']

    candidates = []
    for nf in sorted(subject_dir.rglob('*.nii.gz')):
        path_lower = str(nf).lower()
        if any(kw in path_lower for kw in exclude_kw):
            continue
        try:
            shape = nib.load(str(nf)).shape
        except Exception:
            continue
        if len(shape) == 4 and shape[3] > 1:
            candidates.append((nf, shape[3]))

    if not candidates:
        return None, 0

    # Prefer the scan with the most timepoints if several qualify.
    candidates.sort(key=lambda c: c[1], reverse=True)
    return str(candidates[0][0]), len(candidates)


def get_tr_seconds(fmri_path: str) -> float:
    """Repetition time in seconds, read from the NIfTI header's 4th-dimension
    zoom — the same source used for every existing row in subjects_df."""
    return float(nib.load(fmri_path).header.get_zooms()[3])


def make_participant_id(original_id: str, group: str) -> str:
    return f"sub-{group}{original_id.replace('_', '')}"


if SUBJECTS_DF_V2_PATH.exists() and not FORCE_RESCAN:
    # ── Fast path: another tab already built this — just load it ─────────
    print(f"✓ Found an already-updated subjects_df on Drive — loading it directly")
    print(f"  (no Drive re-scan needed — this was built by another tab/run):")
    print(f"  {SUBJECTS_DF_V2_PATH}")
    subjects_df = pd.read_csv(SUBJECTS_DF_V2_PATH)
    print(f"\n  Total subjects: {len(subjects_df)}  "
          f"(CN: {(subjects_df['group']=='CN').sum()}  |  "
          f"EMCI: {(subjects_df['group']=='EMCI').sum()})")
    print(f"\n  If more subjects were added since this file was built, set")
    print(f"  FORCE_RESCAN = True above and re-run this cell to refresh it.")

else:
    # ── Slow path: scan Drive and build the updated subjects_df ────────────
    # ── Scan Drive for every subject directory that currently exists ───────
    all_dirs = []
    for group in ['CN', 'EMCI']:
        group_dir = NIFTI_ROOT / group
        if not group_dir.exists():
            continue
        for d in sorted(group_dir.iterdir()):
            if d.is_dir():
                all_dirs.append((d.name, group))

    n_cn_dirs   = sum(1 for _, g in all_dirs if g == 'CN')
    n_emci_dirs = sum(1 for _, g in all_dirs if g == 'EMCI')
    print(f"Found {len(all_dirs)} subject directories on Drive "
          f"({n_cn_dirs} CN / {n_emci_dirs} EMCI)")

    existing_ids = set(subjects_df['original_id'].astype(str))
    print(f"Already in subjects_df: {len(existing_ids)}")

    # ── Build rows only for directories not already in subjects_df ─────────
    new_rows             = []
    no_fmri_found        = []
    multi_candidate_subj = []

    for original_id, group in all_dirs:
        if original_id in existing_ids:
            continue   # already present — left completely untouched

        fmri_path, n_candidates = find_fmri_for_subject(original_id, group, NIFTI_ROOT)
        if fmri_path is None:
            no_fmri_found.append(original_id)
            continue
        if n_candidates > 1:
            multi_candidate_subj.append((original_id, n_candidates))

        try:
            tr = get_tr_seconds(fmri_path)
        except Exception as e:
            no_fmri_found.append(f"{original_id} (TR read failed: {str(e)[:60]})")
            continue

        new_rows.append({
            'original_id'   : original_id,
            'participant_id': make_participant_id(original_id, group),
            'group'         : group,
            'label_encoded' : GROUP_LABELS[group],
            'filepath'      : fmri_path,
            'tr'            : tr,
        })

    new_df = pd.DataFrame(new_rows, columns=subjects_df.columns)
    subjects_df_updated = pd.concat([subjects_df, new_df], ignore_index=True)

    # ── Save to a NEW file — original subjects_df.csv is never overwritten ──
    subjects_df_updated.to_csv(SUBJECTS_DF_V2_PATH, index=False)

    # Everything from CELL 6 onward uses the updated frame.
    subjects_df = subjects_df_updated

    print(f"\n✓ Updated subjects_df saved → {SUBJECTS_DF_V2_PATH}")
    print(f"  Original file left untouched → {SUBJECTS_DF_PATH}")
    print(f"  Previous total : {len(existing_ids)}")
    print(f"  Newly added    : {len(new_rows)}")
    print(f"  New total      : {len(subjects_df)}  "
          f"(CN: {(subjects_df['group']=='CN').sum()}  |  "
          f"EMCI: {(subjects_df['group']=='EMCI').sum()})")

    if no_fmri_found:
        print(f"\n⚠ {len(no_fmri_found)} new subject folder(s) had no usable fMRI "
              f"detected — these are NOT included yet. This likely means their "
              f"DICOM→NIfTI conversion needs another look:")
        for sid in no_fmri_found[:15]:
            print(f"    {sid}")
        if len(no_fmri_found) > 15:
            print(f"    ... and {len(no_fmri_found) - 15} more")

    if multi_candidate_subj:
        print(f"\nNote: {len(multi_candidate_subj)} new subject(s) had more than "
              f"one candidate fMRI scan — the longest (most volumes) was kept. "
              f"Worth a manual check:")
        for sid, n in multi_candidate_subj[:10]:
            print(f"    {sid}: {n} candidates")

    print()
    print(new_df.head(3).to_string(index=False) if len(new_df) else "(no new rows)")


✓ Found an already-updated subjects_df on Drive — loading it directly
  (no Drive re-scan needed — this was built by another tab/run):
  /content/drive/MyDrive/enhanced_pipeline/subjects_df_v2.csv

  Total subjects: 445  (CN: 223  |  EMCI: 222)

  If more subjects were added since this file was built, set
  FORCE_RESCAN = True above and re-run this cell to refresh it.


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 6 — Discover T1w paths by scanning Preprocessed_ADNI_NIFTI
# ──────────────────────────────────────────────────────────────────────

def find_t1w_for_subject(original_id: str, group: str, nifti_root: Path) -> str | None:
    """
    Search for a T1w NIfTI file for one subject under:
        <nifti_root>/<group>/<original_id>/T1w/*.nii.gz

    The T1w subfolder is named 'T1w' in the existing folder structure
    (as documented in the project context). Falls back to a recursive
    search for MPRAGE or T1w-like filenames if the folder is absent.

    Returns the path as a string, or None if not found.
    """
    subject_dir = nifti_root / group / original_id

    if not subject_dir.exists():
        return None

    # Primary: dedicated T1w/ subfolder
    t1w_dir = subject_dir / 'T1w'
    if t1w_dir.exists():
        candidates = sorted(t1w_dir.rglob('*.nii.gz'))
        if candidates:
            return str(candidates[0])

    # Fallback: scan all subfolders for MPRAGE / T1w filenames
    t1w_keywords = ['mprage', 't1w', 't1_', 'bravo', 'spgr', 'tfl']
    for nf in sorted(subject_dir.rglob('*.nii.gz')):
        name_lower = nf.name.lower()
        if any(kw in name_lower for kw in t1w_keywords):
            # Make sure it is 3D (not the 4D fMRI)
            try:
                shape = nib.load(str(nf)).shape
                if len(shape) == 3:
                    return str(nf)
            except Exception:
                continue

    return None


print("Scanning for T1w files...")
print("(This scans filenames only — no images are loaded — should take < 1 min)")
print()

t1w_paths   = []
missing_t1w = []

for _, row in subjects_df.iterrows():
    sid   = str(row['original_id'])
    group = str(row['group'])
    path  = find_t1w_for_subject(sid, group, NIFTI_ROOT)
    t1w_paths.append(path)
    if path is None:
        missing_t1w.append(sid)

subjects_df = subjects_df.copy()
subjects_df['t1w_path'] = t1w_paths

# NEW: explicit per-subject registration method, used everywhere downstream
# (CELL 7 confound check, CELL 9 partitioning, CELL 14 dispatch, CELL 15 QC).
subjects_df['registration_method'] = np.where(
    subjects_df['t1w_path'].notna(), 'T1w_mediated', 'Direct_EPI_MNI'
)

found = subjects_df['t1w_path'].notna().sum()
print(f"✓ T1w discovery complete")
print(f"  Found    : {found} / {len(subjects_df)}  → will use T1w-mediated registration")
print(f"  Missing  : {len(missing_t1w)}  → will use direct EPI→MNI fallback (CELL 13)")
print(f"  (No subjects are skipped or dropped for lack of T1w — see CELL 7 for the")
print(f"   confound check on whether this split lines up with diagnosis group.)")
if missing_t1w:
    print(f"\n  Subjects using the fallback (first 10):")
    for sid in missing_t1w[:10]:
        print(f"    {sid}")
    if len(missing_t1w) > 10:
        print(f"    ... and {len(missing_t1w) - 10} more")


Scanning for T1w files...
(This scans filenames only — no images are loaded — should take < 1 min)

✓ T1w discovery complete
  Found    : 158 / 445  → will use T1w-mediated registration
  Missing  : 287  → will use direct EPI→MNI fallback (CELL 13)
  (No subjects are skipped or dropped for lack of T1w — see CELL 7 for the
   confound check on whether this split lines up with diagnosis group.)

  Subjects using the fallback (first 10):
    005_S_0602
    007_S_1222
    002_S_6007
    002_S_6009
    002_S_6030
    002_S_6053
    002_S_6066
    002_S_6103
    002_S_6404
    002_S_6456
    ... and 277 more


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 7 — Confound check: is T1w availability associated with diagnosis?
# ──────────────────────────────────────────────────────────────────────
#
# Mixing two registration pipelines (T1w-mediated vs direct EPI→MNI) in one
# dataset is only safe if missing-T1w is NOT systematically associated with
# group (CN/EMCI). If it is, registration method becomes a confound that a
# downstream classifier could latch onto instead of real neurobiology — the
# same kind of issue as the TR-heterogeneity and data-leakage problems fixed
# earlier in this project.
#
# This cell:
#   1. Cross-tabulates group × registration method.
#   2. Runs a chi-square test of independence.
#   3. Prints a clear warning if the association looks non-random.

from scipy.stats import chi2_contingency

ct = pd.crosstab(subjects_df['group'], subjects_df['registration_method'])
print("Group × Registration-method cross-tab:")
print(ct)
print()

if ct.shape[0] > 1 and ct.shape[1] > 1 and ct.values.min() > 0:
    chi2, p, dof, _ = chi2_contingency(ct)
    print(f"Chi-square test of independence: χ²={chi2:.3f}, dof={dof}, p={p:.4f}")
    if p < 0.05:
        print("\n⚠ WARNING: T1w availability is significantly associated with group.")
        print("  Registration method is a potential confound — report this explicitly")
        print("  in your methods/limitations, and check CELL 15 to see whether measured")
        print("  registration quality also differs by method/group.")
    else:
        print("\n✓ No significant association detected — registration method is")
        print("  unlikely to be a strong confound, but still keep tracking it downstream")
        print("  (e.g. as a covariate, or reported alongside your results).")
else:
    print("One group/method combination has zero subjects in it —")
    print("cannot run a chi-square test. Inspect the cross-tab above manually:")
    print("if one group has disproportionately more missing T1w, treat registration")
    print("method as a confound regardless of a formal test.")

print(f"\nOverall registration method breakdown:")
print(subjects_df['registration_method'].value_counts())


Group × Registration-method cross-tab:
registration_method  Direct_EPI_MNI  T1w_mediated
group                                            
CN                              141            82
EMCI                            146            76

Chi-square test of independence: χ²=0.212, dof=1, p=0.6454

✓ No significant association detected — registration method is
  unlikely to be a strong confound, but still keep tracking it downstream
  (e.g. as a covariate, or reported alongside your results).

Overall registration method breakdown:
registration_method
Direct_EPI_MNI    287
T1w_mediated      158
Name: count, dtype: int64


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Download MNI152 2mm brain template (once, shared across all sessions)
# ─────────────────────────────────────────────────────────────────────────

def download_mni_template(dest: Path) -> Path:
    """
    Download MNI152_T1_2mm_brain from TemplateFlow CDN (OSF fallback).
    Skips download if file already exists — safe to re-run.
    All sessions share the same file; no write conflict (read-only after creation).
    """
    if dest.exists():
        print(f"✓ MNI template already on Drive ({dest.name})")
        return dest

    mirrors = [
        ("https://templateflow.s3.amazonaws.com/tpl-MNI152NLin2009cAsym/"
         "tpl-MNI152NLin2009cAsym_res-02_T1w.nii.gz"),
        "https://osf.io/download/jkzpu/",
    ]
    for url in mirrors:
        try:
            print(f"  Downloading from {url[:65]}...")
            urllib.request.urlretrieve(url, dest)
            print(f"✓ Saved → {dest.name}")
            return dest
        except Exception as e:
            print(f"  Mirror failed: {e}")

    raise RuntimeError(
        "All MNI mirrors failed.\n"
        f"Upload MNI152_T1_2mm_brain.nii.gz manually to {dest.parent}"
    )

MNI_PATH = download_mni_template(MNI_PATH)

_t = ants.image_read(str(MNI_PATH))
print(f"  Shape  : {_t.shape}")
print(f"  Spacing: {_t.spacing}")
del _t

✓ MNI template already on Drive (MNI152_T1_2mm_brain.nii.gz)
  Shape  : (91, 109, 91)
  Spacing: (2.0, 2.0, 2.0)


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 9 — Build or reload subject partition for this session
# ──────────────────────────────────────────────────────────────────────

def build_partitions_if_needed(df, n_sessions, seed, reg_root):
    """
    Write one CSV per session if they don't already exist OR if subjects_df
    has grown since the partitions were last built (e.g. after CELL 5 added
    newly converted subjects).

    Subjects are shuffled before splitting so each session gets a balanced
    mix of scan types (avoids one session getting all slow multiband subjects).
    The first tab to run this creates all CSVs; subsequent tabs just read them.

    Rebuilding is safe even for already-registered subjects: it only changes
    which session CSV a subject's row lives in, never touches any registered
    output file, and CELL 11 reconciles already-completed subjects against
    their output files on Drive regardless of which session they land in
    after a rebuild.
    """
    csv_paths = [reg_root / f'session_{i}_subjects.csv' for i in range(n_sessions)]
    all_exist = all(p.exists() for p in csv_paths)

    if all_exist:
        covered_ids = set()
        for p in csv_paths:
            covered_ids.update(pd.read_csv(p)['original_id'].astype(str))
        current_ids = set(df['original_id'].astype(str))
        new_ids     = current_ids - covered_ids
        if not new_ids:
            print("  Partition CSVs already exist on Drive and cover all current subjects — skipping rebuild")
            return
        print(f"  {len(new_ids)} subject(s) in subjects_df aren't covered by the existing "
              f"partitions — rebuilding all {n_sessions} partitions to include them.")
        print("  (Already-registered subjects are unaffected — CELL 11 reconciles them")
        print("   against their output files on Drive regardless of the new split.)")

    print(f"  Building {n_sessions} partitions (seed={seed})...")
    shuffled   = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    partitions = np.array_split(shuffled, n_sessions)
    for i, part in enumerate(partitions):
        out = reg_root / f'session_{i}_subjects.csv'
        part.to_csv(out, index=False)
        print(f"    Session {i}: {len(part)} subjects → {out.name}")

build_partitions_if_needed(subjects_df, N_SESSIONS, PARTITION_SEED, REG_ROOT)

# Load only this session's slice (includes t1w_path column we just built)
session_df = pd.read_csv(SESSION_CSV)

# Re-attach t1w_path AND registration_method — partition CSVs were written
# from the base subjects_df which did not have these columns yet, so we
# merge them back in now.
t1w_lookup    = subjects_df.set_index('original_id')['t1w_path'].to_dict()
method_lookup = subjects_df.set_index('original_id')['registration_method'].to_dict()
session_df['t1w_path']            = session_df['original_id'].map(t1w_lookup)
session_df['registration_method'] = session_df['original_id'].map(method_lookup)

# ── Safety check: catch a stale/incomplete local subjects_df immediately ──
# If this tab's subjects_df doesn't yet include CELL 6's merged subjects
# (e.g. CELL 6 wasn't re-run in THIS tab even though another tab already
# rebuilt the shared partition files), some original_ids in session_df won't
# be known here and the .map() calls above silently produce NaN. Left
# unchecked, CELL 15 would then treat those subjects as "no T1w" and run
# them through the weaker direct EPI→MNI fallback even if they actually
# have a T1w scan — a silent registration-quality regression. Fail loudly
# instead.
unknown_ids = set(session_df['original_id'].astype(str)) - set(subjects_df['original_id'].astype(str))
if unknown_ids:
    raise RuntimeError(
        f"{len(unknown_ids)} subject(s) assigned to this session aren't in "
        f"THIS tab's local subjects_df (e.g. {sorted(unknown_ids)[:5]}).\n"
        f"This almost always means CELL 6 (and therefore CELL 7) wasn't run "
        f"in THIS tab/runtime before this cell — each tab has its own "
        f"independent Python state, so CELL 6 must be (re-)run in every tab, "
        f"not just once. Re-run CELL 4 → 5 → 6 → 7 → 8 → 9 in this tab, then retry."
    )

print(f"\n✓ Session {SESSION_ID}: {len(session_df)} subjects assigned")
print(f"  T1w-mediated   : {(session_df['registration_method'] == 'T1w_mediated').sum()}")
print(f"  Direct EPI→MNI : {(session_df['registration_method'] == 'Direct_EPI_MNI').sum()}")
print(session_df[['original_id', 'group', 'tr', 'registration_method']].head(3).to_string(index=False))


  Partition CSVs already exist on Drive and cover all current subjects — skipping rebuild

✓ Session 1: 148 subjects assigned
  T1w-mediated   : 49
  Direct EPI→MNI : 99
original_id group       tr registration_method
 082_S_6197  EMCI 3.000000      Direct_EPI_MNI
 018_S_6351    CN 2.999998      Direct_EPI_MNI
 027_S_2219  EMCI 3.000000        T1w_mediated


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 10 — Progress checkpoint helpers
# ──────────────────────────────────────────────────────────────────────

def load_progress(path: Path) -> dict:
    """Load per-session progress JSON, or return empty structure if first run.
    Back-compat: older progress files (pre-T1w-fallback) won't have a
    'method' key — add it on load so downstream code can rely on it.
    """
    if path.exists():
        with open(path) as f:
            progress = json.load(f)
        progress.setdefault('method', {})    # {original_id: 'T1w_mediated' | 'Direct_EPI_MNI'}
        progress.setdefault('skipped', [])
        progress.setdefault('failed', {})
        progress.setdefault('completed', [])
        return progress
    return {
        'session_id'  : SESSION_ID,
        'completed'   : [],
        'method'      : {},   # {original_id: 'T1w_mediated' | 'Direct_EPI_MNI'}
        'failed'      : {},
        'skipped'     : [],
        'last_updated': None,
    }

def save_progress(progress: dict, path: Path) -> None:
    """Atomic save: write to .tmp then rename — safe if Colab dies mid-write."""
    progress['last_updated'] = datetime.now().isoformat()
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'w') as f:
        json.dump(progress, f, indent=2)
    tmp.rename(path)

def registered_fmri_path(original_id: str, group: str) -> Path:
    """Canonical output path — unique per subject, no cross-session conflicts."""
    return REG_ROOT / group / f"{original_id}_registered_fmri.nii.gz"

def is_already_done(original_id: str, group: str, progress: dict) -> bool:
    out = registered_fmri_path(original_id, group)
    return (original_id in progress['completed']) and out.exists()

def print_status(progress: dict, df: pd.DataFrame) -> None:
    total     = len(df)
    done      = len(progress['completed'])
    failed    = len(progress['failed'])
    skipped   = len(progress['skipped'])
    remaining = total - done - failed - skipped
    methods   = progress.get('method', {})
    n_t1w     = sum(1 for v in methods.values() if v == 'T1w_mediated')
    n_direct  = sum(1 for v in methods.values() if v == 'Direct_EPI_MNI')
    print(f"  Session {SESSION_ID} ─────────────────────────────")
    print(f"  Assigned  : {total}")
    print(f"  Completed : {done}    (T1w-mediated: {n_t1w}  |  Direct EPI→MNI: {n_direct})")
    print(f"  Failed    : {failed}")
    print(f"  Skipped   : {skipped}  (legacy/edge-case skips, if any)")
    print(f"  Remaining : {remaining}")
    if progress.get('last_updated'):
        print(f"  Last save : {progress['last_updated'][:19]}")
    if progress.get('failed'):
        print("  Failed subjects (first 3):")
        for sid, err in list(progress['failed'].items())[:3]:
            print(f"    {sid}: {err[:80]}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 11 — Reconcile subjects already registered in a previous run
# ──────────────────────────────────────────────────────────────────────
#
# If you already registered a batch of subjects — all T1w-mediated, from an
# earlier run of this notebook — their output files exist on Drive but may
# not be listed in THIS session's progress file (e.g. after repartitioning
# subjects, restoring a fresh runtime, or moving to this updated notebook).
#
# This cell finds those files and marks them 'completed' with
# method='T1w_mediated', WITHOUT touching or re-registering them — this is
# the "keep previously-registered samples as-is" decision from the methods
# discussion: re-running the weaker direct EPI→MNI path over good existing
# T1w-mediated registrations would only degrade data you already trust.

progress = load_progress(PROGRESS_FILE)
n_recovered = 0

for _, row in session_df.iterrows():
    sid   = str(row['original_id'])
    group = str(row['group'])
    out   = registered_fmri_path(sid, group)

    already_listed = sid in progress['completed']
    if not already_listed and out.exists():
        progress['completed'].append(sid)
        # Files predating this notebook version were necessarily produced by
        # the original T1w-mediated pipeline (the only path that existed then).
        progress['method'][sid] = 'T1w_mediated'
        n_recovered += 1

if n_recovered:
    save_progress(progress, PROGRESS_FILE)
    print(f"✓ Recovered {n_recovered} previously-registered subject(s) into progress")
    print("  (marked complete, method=T1w_mediated, NOT re-registered or modified)")
else:
    print("No previously-registered files found outside the progress log — nothing to recover.")

print()
print_status(progress, session_df)


✓ Recovered 28 previously-registered subject(s) into progress
  (marked complete, method=T1w_mediated, NOT re-registered or modified)

  Session 1 ─────────────────────────────
  Assigned  : 148
  Completed : 92    (T1w-mediated: 28  |  Direct EPI→MNI: 0)
  Failed    : 0
  Skipped   : 1  (legacy/edge-case skips, if any)
  Remaining : 55
  Last save : 2026-06-19T22:37:55


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 12 — Core per-subject registration function
# ─────────────────────────────────────────────────────────────────────────

def register_subject_t1w(
    original_id: str,
    fmri_path: str,
    t1w_path: str,
    group: str,
    mni_template_path: Path,
    tr: float,
) -> Path:
    """
    Register one subject's fMRI to MNI space via their T1w structural image.
    Used only when a T1w scan is available for this subject — see
    register_subject_direct() in CELL 13 for the no-T1w fallback.

    Transform chain
    ───────────────
        fMRI (native 4D)
            ↓  Rigid (6 DOF) — corrects positioning between fMRI and T1w sessions
        T1w (subject space)
            ↓  SyN (non-linear) — warps individual anatomy to MNI atlas
        MNI152 (standard space)

    The combined transform is applied volume-by-volume to the full 4D fMRI.
    Memory is freed after each volume — essential for 976-vol multiband scans.

    Parameters
    ──────────
    original_id       : subject ID as in subjects_df, e.g. '002_S_0413'
    fmri_path         : path to 4D NIfTI fMRI (native space)
    t1w_path          : path to 3D T1w NIfTI (subject space)
    group             : 'CN' or 'EMCI'
    mni_template_path : path to MNI152_T1_2mm_brain.nii.gz
    tr                : repetition time in seconds
    """
    out_path = registered_fmri_path(original_id, group)

    # ── Load structural images ────────────────────────────────────────────────
    t1w_ants = ants.image_read(str(t1w_path))
    mni_ants = ants.image_read(str(mni_template_path))

    # ── Load 4D fMRI ─────────────────────────────────────────────────────────
    fmri_nib  = nib.load(str(fmri_path))
    fmri_data = fmri_nib.get_fdata(dtype=np.float32)   # (X, Y, Z, T)
    n_vols    = fmri_data.shape[3]

    # ── Mean fMRI reference volume (skip first 4 dummies) ────────────────────
    start_vol = 4 if n_vols > 10 else 0
    mean_vol  = fmri_data[:, :, :, start_vol:].mean(axis=3).astype(np.float32)

    sform  = fmri_nib.header.get_sform()
    zooms  = np.array(fmri_nib.header.get_zooms()[:3], dtype=float)
    mean_fmri_ants = ants.from_numpy(
        mean_vol,
        origin    = list(sform[:3, 3]),
        spacing   = list(zooms),
        direction = sform[:3, :3] / zooms,
    )

    # ── Step 1: Rigid — mean fMRI → T1w ──────────────────────────────────────
    print(f"    [1/4] Rigid: mean fMRI → T1w ...")
    fmri_to_t1w = ants.registration(
        fixed=t1w_ants, moving=mean_fmri_ants,
        type_of_transform='Rigid', verbose=False,
    )

    # ── Step 2: T1w → MNI registration ──────────────────────────────────────
    # Use the full antsRegistrationSyN[s] script: rigid + affine + deformable.
    # This is the correct ANTsPy string (verified from ants 0.6.3 docs).
    #
    # Two options — swap REG_TYPE to choose:
    #   'antsRegistrationSyN[s]'      ~ 35-45 min/subject  (highest quality)
    #   'antsRegistrationSyNQuick[s]' ~ 15-20 min/subject  (faster, slightly less accurate)
    REG_TYPE = 'antsRegistrationSyN[s]'
    print(f"    [2/4] T1w → MNI ({REG_TYPE}) ...")
    t1w_to_mni = ants.registration(
        fixed=mni_ants, moving=t1w_ants,
        type_of_transform=REG_TYPE, verbose=False,
    )

    # ── Step 3: Apply combined transform to all fMRI volumes ──────────────────
    combined = (
        t1w_to_mni['fwdtransforms'] +   # SyN warp + affine (T1w → MNI)
        fmri_to_t1w['fwdtransforms']     # Rigid (fMRI → T1w)
    )
    out_shape     = mni_ants.shape
    registered_4d = np.zeros((*out_shape, n_vols), dtype=np.float32)

    print(f"    [3/4] Applying transform to {n_vols} volumes ...")
    for t in range(n_vols):
        vol = ants.from_numpy(
            fmri_data[:, :, :, t],
            origin=mean_fmri_ants.origin,
            spacing=mean_fmri_ants.spacing,
            direction=mean_fmri_ants.direction,
        )
        warped = ants.apply_transforms(
            fixed=mni_ants, moving=vol,
            transformlist=combined, interpolator='linear',
        )
        registered_4d[:, :, :, t] = warped.numpy()
        del vol, warped
        if t % 50 == 0:
            gc.collect()

    # ── Step 4: Save registered 4D NIfTI ─────────────────────────────────────
    print(f"    [4/4] Saving → {out_path.name}")
    affine = np.eye(4)
    affine[:3, :3] = np.diag(mni_ants.spacing)
    affine[:3,  3] = mni_ants.origin
    out_img = nib.Nifti1Image(registered_4d, affine)
    out_img.header.set_zooms((*mni_ants.spacing, tr))
    nib.save(out_img, str(out_path))

    del fmri_data, mean_vol, registered_4d
    del fmri_to_t1w, t1w_to_mni, t1w_ants, mni_ants, mean_fmri_ants
    gc.collect()
    return out_path

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 13 — Core per-subject registration function: DIRECT EPI → MNI
#           (fallback used only when no T1w structural scan is available)
# ──────────────────────────────────────────────────────────────────────

def register_subject_direct(
    original_id: str,
    fmri_path: str,
    group: str,
    mni_template_path: Path,
    tr: float,
) -> Path:
    """
    Register one subject's fMRI directly to MNI space, with NO T1w step.

    Transform chain
    ─────────────
        Mean fMRI (mean over time, native space)
            ↓  antsRegistrationSyN[s] (rigid + affine + non-linear, single call)
        MNI152 (standard space)

    This is a weaker registration than the T1w-mediated path in CELL 12: EPI
    images have lower spatial resolution, more distortion, and worse tissue
    contrast than a T1w structural, so direct EPI→MNI alignment is typically
    less accurate — especially around the hippocampus / medial temporal lobe,
    which matters for CN vs EMCI classification.

    Use this ONLY as a fallback for subjects missing a T1w scan, and always
    keep tracking `registration_method` (set by CELL 6, checked in CELL 7,
    and reported in CELL 15) so you can confirm downstream that classifier
    performance isn't being driven by which registration pipeline a subject
    went through rather than real neurobiology.

    Parameters mirror register_subject_t1w() in CELL 12, minus t1w_path.
    """
    out_path = registered_fmri_path(original_id, group)

    # ── Load structural target ────────────────────────────────────────────
    mni_ants = ants.image_read(str(mni_template_path))

    # ── Load 4D fMRI ──────────────────────────────────────────────────────
    fmri_nib  = nib.load(str(fmri_path))
    fmri_data = fmri_nib.get_fdata(dtype=np.float32)   # (X, Y, Z, T)
    n_vols    = fmri_data.shape[3]

    # ── Mean fMRI reference volume (skip first 4 dummies) ────────────
    start_vol = 4 if n_vols > 10 else 0
    mean_vol  = fmri_data[:, :, :, start_vol:].mean(axis=3).astype(np.float32)

    sform  = fmri_nib.header.get_sform()
    zooms  = np.array(fmri_nib.header.get_zooms()[:3], dtype=float)
    mean_fmri_ants = ants.from_numpy(
        mean_vol,
        origin    = list(sform[:3, 3]),
        spacing   = list(zooms),
        direction = sform[:3, :3] / zooms,
    )

    # ── Single step: mean fMRI → MNI directly (rigid + affine + SyN) ─────
    # Same registration string as the T1w-mediated path's structural→MNI step,
    # just applied straight from EPI space since there is no T1w to mediate.
    REG_TYPE = 'antsRegistrationSyN[s]'
    print(f"    [1/3] Direct EPI → MNI ({REG_TYPE}) — no T1w available ...")
    fmri_to_mni = ants.registration(
        fixed=mni_ants, moving=mean_fmri_ants,
        type_of_transform=REG_TYPE, verbose=False,
    )

    combined  = fmri_to_mni['fwdtransforms']
    out_shape = mni_ants.shape
    registered_4d = np.zeros((*out_shape, n_vols), dtype=np.float32)

    print(f"    [2/3] Applying transform to {n_vols} volumes ...")
    for t in range(n_vols):
        vol = ants.from_numpy(
            fmri_data[:, :, :, t],
            origin=mean_fmri_ants.origin,
            spacing=mean_fmri_ants.spacing,
            direction=mean_fmri_ants.direction,
        )
        warped = ants.apply_transforms(
            fixed=mni_ants, moving=vol,
            transformlist=combined, interpolator='linear',
        )
        registered_4d[:, :, :, t] = warped.numpy()
        del vol, warped
        if t % 50 == 0:
            gc.collect()

    # ── Save registered 4D NIfTI ─────────────────────────────────
    print(f"    [3/3] Saving → {out_path.name}")
    affine = np.eye(4)
    affine[:3, :3] = np.diag(mni_ants.spacing)
    affine[:3,  3] = mni_ants.origin
    out_img = nib.Nifti1Image(registered_4d, affine)
    out_img.header.set_zooms((*mni_ants.spacing, tr))
    nib.save(out_img, str(out_path))

    del fmri_data, mean_vol, registered_4d
    del fmri_to_mni, mni_ants, mean_fmri_ants
    gc.collect()
    return out_path


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 14 — Registration loop (resumable — re-run this cell after any disconnect)
# ──────────────────────────────────────────────────────────────────────

def run_session(session_df: pd.DataFrame) -> dict:
    """
    Process all subjects assigned to this session.
    - Skips already-completed subjects automatically (reads per-session progress file).
    - Dispatches each remaining subject to the T1w-mediated path (CELL 12) if a
      T1w scan was found, otherwise to the direct EPI→MNI fallback (CELL 13).
      No subject is dropped for lack of T1w.
    - Saves progress (including which method was used) to Drive after every subject.
    Safe to interrupt and re-run — resumes from where it stopped.
    """
    progress = load_progress(PROGRESS_FILE)
    print_status(progress, session_df)
    print()

    total = len(session_df)
    for idx, (_, row) in enumerate(session_df.iterrows()):
        sid    = str(row['original_id'])
        group  = str(row['group'])
        fmri_p = str(row['filepath'])
        t1w_p  = row['t1w_path']
        tr     = float(row['tr'])

        has_t1w = pd.notna(t1w_p) and Path(str(t1w_p)).exists()
        method  = 'T1w_mediated' if has_t1w else 'Direct_EPI_MNI'

        # ── Already done ──────────────────────────────────────────────
        if is_already_done(sid, group, progress):
            done_method = progress['method'].get(sid, 'unknown')
            print(f"  [{idx+1}/{total}] {sid} — already done ✓  ({done_method})")
            continue

        # ── fMRI file missing (should not happen for processed subjects) ──────
        if not Path(fmri_p).exists():
            msg = f"fMRI not found: {fmri_p}"
            print(f"  [{idx+1}/{total}] {sid} — FAILED ({msg})")
            progress['failed'][sid] = msg
            save_progress(progress, PROGRESS_FILE)
            continue

        # ── Register (dispatch by T1w availability — nobody gets skipped) ─────
        scan_tag = 'MB' if tr < 1.0 else 'std'
        print(f"\n  [{idx+1}/{total}] {sid}  group={group}  TR={tr}s  ({scan_tag})  method={method}")
        t0 = datetime.now()
        try:
            if has_t1w:
                out = register_subject_t1w(
                    original_id       = sid,
                    fmri_path         = fmri_p,
                    t1w_path          = str(t1w_p),
                    group             = group,
                    mni_template_path = MNI_PATH,
                    tr                = tr,
                )
            else:
                out = register_subject_direct(
                    original_id       = sid,
                    fmri_path         = fmri_p,
                    group             = group,
                    mni_template_path = MNI_PATH,
                    tr                = tr,
                )
            elapsed = round((datetime.now() - t0).total_seconds() / 60, 1)
            print(f"  ✓ {sid}  {elapsed} min → {out.name}  [{method}]")
            progress['completed'].append(sid)
            progress['method'][sid] = method
        except Exception as e:
            elapsed = round((datetime.now() - t0).total_seconds() / 60, 1)
            msg = f"{type(e).__name__}: {str(e)[:300]}"
            print(f"  ✗ {sid}  {elapsed} min  FAILED: {msg}")
            progress['failed'][sid] = msg

        # ── Save after EVERY subject (atomic write) ───────────────────
        save_progress(progress, PROGRESS_FILE)
        gc.collect()

    print("\n" + "═"*55)
    print(f"Session {SESSION_ID} — loop finished")
    print_status(progress, session_df)
    return progress

session_progress = run_session(session_df)


  Session 1 ─────────────────────────────
  Assigned  : 148
  Completed : 92    (T1w-mediated: 28  |  Direct EPI→MNI: 0)
  Failed    : 0
  Skipped   : 1  (legacy/edge-case skips, if any)
  Remaining : 55
  Last save : 2026-06-19T22:37:55


  [1/148] 082_S_6197  group=EMCI  TR=3.0s  (std)  method=Direct_EPI_MNI
    [1/3] Direct EPI → MNI (antsRegistrationSyN[s]) — no T1w available ...
    [2/3] Applying transform to 197 volumes ...
    [3/3] Saving → 082_S_6197_registered_fmri.nii.gz
  ✓ 082_S_6197  9.0 min → 082_S_6197_registered_fmri.nii.gz  [Direct_EPI_MNI]

  [2/148] 018_S_6351  group=CN  TR=2.999997854232788s  (std)  method=Direct_EPI_MNI
    [1/3] Direct EPI → MNI (antsRegistrationSyN[s]) — no T1w available ...
    [2/3] Applying transform to 197 volumes ...
    [3/3] Saving → 018_S_6351_registered_fmri.nii.gz
  ✓ 018_S_6351  8.9 min → 018_S_6351_registered_fmri.nii.gz  [Direct_EPI_MNI]
  [3/148] 027_S_2219 — already done ✓  (unknown)

  [4/148] 021_S_6910  group=CN  TR=3.0s  (std

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — Full registration quality check (all completed subjects)
# ─────────────────────────────────────────────────────────────────────────────
#
# Evaluates EVERY subject completed in this session (not just a sample).
# For each subject, computes three metrics:
#
#   CC  — Cross-Correlation between registered mean fMRI and MNI template.
#          Measures overall intensity alignment.  Target: > 0.70
#
#   MI  — Mutual Information (normalized).
#          Measures statistical dependency between image intensities.
#          More robust than CC when contrast differs between modalities.
#          Target: > 0.30
#
#   DIV — Mean absolute difference in the brain mask region (divergence).
#          Lower = better alignment.  Target: < 0.15
#
# Quality tiers:
#   GOOD     : CC > 0.70
#   MARGINAL : CC 0.50 – 0.70  (may need re-registration with full SyN)
#   POOR     : CC < 0.50        (likely failed — flag for manual inspection)
#
# Outputs:
#   1. Per-subject table printed to console
#   2. Summary statistics (mean, std, min, max per metric)
#   3. Distribution histogram of CC scores saved to Drive
#   4. Visual grid of axial slices for ALL subjects saved to Drive
#      (3 panels per subject: MNI | registered mean fMRI | abs difference)
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def compute_registration_metrics(mni_vol, reg_mean_vol):
    """
    Compute CC, normalized MI, and mean absolute divergence between
    the MNI template and a registered mean fMRI volume.

    Both inputs are normalised to [0,1] before metric computation.

    Returns
    -------
    cc  : float  Cross-correlation in brain mask
    mi  : float  Normalized mutual information
    div : float  Mean absolute difference in brain mask
    """
    norm = lambda v: (v - v.min()) / (v.max() - v.min() + 1e-8)

    s   = tuple(min(a, b) for a, b in zip(mni_vol.shape, reg_mean_vol.shape))
    mc  = norm(mni_vol[:s[0], :s[1], :s[2]])
    rc  = norm(reg_mean_vol[:s[0], :s[1], :s[2]])

    # Brain mask: non-background voxels in MNI template
    mask = mc > 0.10
    if mask.sum() < 100:
        return float('nan'), float('nan'), float('nan'), mc, rc, s

    mc_m = mc[mask]
    rc_m = rc[mask]

    # ── Cross-Correlation ──────────────────────────────────────────────────
    cc = float(np.corrcoef(mc_m, rc_m)[0, 1])

    # ── Normalized Mutual Information ─────────────────────────────────────
    # Approximated via 2D histogram — standard approach used in FSL/SPM.
    bins = 64
    hist_2d, _, _ = np.histogram2d(mc_m, rc_m, bins=bins,
                                    range=[[0,1],[0,1]])
    hist_2d = hist_2d / hist_2d.sum()                        # joint probability
    p_mc = hist_2d.sum(axis=1)                               # marginal P(mc)
    p_rc = hist_2d.sum(axis=0)                               # marginal P(rc)

    # Shannon entropy: H = -sum(p * log(p))
    eps  = 1e-12
    h_mc = -np.sum(p_mc[p_mc > eps] * np.log(p_mc[p_mc > eps]))
    h_rc = -np.sum(p_rc[p_rc > eps] * np.log(p_rc[p_rc > eps]))
    flat = hist_2d[hist_2d > eps]
    h_joint = -np.sum(flat * np.log(flat))
    # NMI = (H(mc) + H(rc)) / H(mc, rc) — ranges [1, 2], rescale to [0, 1]
    mi   = float((h_mc + h_rc) / (h_joint + eps) - 1.0)

    # ── Mean Absolute Divergence ───────────────────────────────────────────
    div  = float(np.mean(np.abs(mc_m - rc_m)))

    return cc, mi, div, mc, rc, s


def full_quality_check(session_df, progress):
    """
    Evaluate registration quality for ALL completed subjects in this session.

    Steps
    ─────
    1. Load MNI template once (reused for all subjects).
    2. For each completed subject: load registered fMRI, compute mean volume,
       compute CC / MI / divergence, store results.
    3. Print per-subject table + aggregate statistics.
    4. Save CC distribution histogram to Drive.
    5. Save visual grid (axial slices) for all subjects to Drive.
       Grid layout: one row per subject, three columns (MNI | registered | diff).
    """

    completed_ids = progress['completed']
    if not completed_ids:
        print("No completed subjects to evaluate yet.")
        return

    n_subjects = len(completed_ids)
    print(f"Evaluating {n_subjects} completed subjects (session {SESSION_ID})...")
    print("Loading MNI template...")
    mni_vol = nib.load(str(MNI_PATH)).get_fdata(dtype=np.float32)

    # ── Per-subject metric collection ─────────────────────────────────────────
    sid_to_group = dict(zip(
        session_df['original_id'].astype(str),
        session_df['group'].astype(str)
    ))
    sid_to_method = progress.get('method', {})

    results = []   # list of dicts, one per subject

    print()
    print(f"{'Subject':<18} {'Group':<6} {'CC':>6} {'NMI':>6} {'DIV':>6}  Quality")
    print("─" * 65)

    for sid in completed_ids:
        group  = sid_to_group.get(sid, 'CN')
        method = sid_to_method.get(sid, 'unknown')
        path   = registered_fmri_path(sid, group)

        if not path.exists():
            print(f"  {sid:<16} {group:<6}  FILE MISSING — skipping")
            continue

        try:
            reg_data = nib.load(str(path)).get_fdata(dtype=np.float32)
            reg_mean = reg_data.mean(axis=3)
            del reg_data

            cc, mi, div, mc_norm, rc_norm, crop_shape = compute_registration_metrics(
                mni_vol, reg_mean
            )

            quality = ("GOOD"     if cc > 0.70 else
                       "MARGINAL" if cc > 0.50 else
                       "POOR")
            icon    = "✓" if quality == "GOOD" else ("⚠" if quality == "MARGINAL" else "✗")

            print(f"  {sid:<16} {group:<6} {cc:>6.3f} {mi:>6.3f} {div:>6.3f}  {icon} {quality}")

            results.append({
                'sid'     : sid,
                'group'   : group,
                'method'  : method,
                'cc'      : cc,
                'mi'      : mi,
                'div'     : div,
                'quality' : quality,
                'mc_norm' : mc_norm,
                'rc_norm' : rc_norm,
                'crop'    : crop_shape,
            })

            del reg_mean, mc_norm, rc_norm
            gc.collect()

        except Exception as e:
            print(f"  {sid:<16} {group:<6}  ERROR: {str(e)[:60]}")

    if not results:
        print("No results to display.")
        return

    # ── Aggregate statistics ───────────────────────────────────────────────────
    cc_vals  = [r['cc']  for r in results if not np.isnan(r['cc'])]
    mi_vals  = [r['mi']  for r in results if not np.isnan(r['mi'])]
    div_vals = [r['div'] for r in results if not np.isnan(r['div'])]

    n_good     = sum(1 for r in results if r['quality'] == 'GOOD')
    n_marginal = sum(1 for r in results if r['quality'] == 'MARGINAL')
    n_poor     = sum(1 for r in results if r['quality'] == 'POOR')

    print("\n" + "═" * 65)
    print(f"  SESSION {SESSION_ID} QUALITY SUMMARY  ({n_subjects} subjects)")
    print("─" * 65)
    print(f"  {'Metric':<10}  {'Mean':>7}  {'Std':>7}  {'Min':>7}  {'Max':>7}")
    print(f"  {'CC':<10}  {np.mean(cc_vals):>7.3f}  {np.std(cc_vals):>7.3f}  "
          f"{np.min(cc_vals):>7.3f}  {np.max(cc_vals):>7.3f}")
    print(f"  {'NMI':<10}  {np.mean(mi_vals):>7.3f}  {np.std(mi_vals):>7.3f}  "
          f"{np.min(mi_vals):>7.3f}  {np.max(mi_vals):>7.3f}")
    print(f"  {'Divergence':<10}  {np.mean(div_vals):>7.3f}  {np.std(div_vals):>7.3f}  "
          f"{np.min(div_vals):>7.3f}  {np.max(div_vals):>7.3f}")
    print("─" * 65)
    print(f"  ✓ GOOD     (CC > 0.70) : {n_good:>3}  ({n_good/len(results)*100:.1f}%)")
    print(f"  ⚠ MARGINAL (CC 0.50–0.70): {n_marginal:>3}  ({n_marginal/len(results)*100:.1f}%)")
    print(f"  ✗ POOR     (CC < 0.50) : {n_poor:>3}  ({n_poor/len(results)*100:.1f}%)")
    print("═" * 65)

    # ── Quality by registration method — the actual confound check ─────────
    methods_present = sorted(set(r['method'] for r in results))
    if len(methods_present) > 1:
        print("\n  Quality by registration method:")
        for m in methods_present:
            vals = [r['cc'] for r in results if r['method'] == m and not np.isnan(r['cc'])]
            if vals:
                print(f"    {m:<18} n={len(vals):>3}  mean CC={np.mean(vals):.3f}  std={np.std(vals):.3f}")
        print("    → If these means differ substantially, registration method may be")
        print("      acting as a confound — cross-check against CELL 7's group association")
        print("      before pooling everything for classification.")

    # Flag poor subjects explicitly for follow-up
    poor_sids = [r['sid'] for r in results if r['quality'] == 'POOR']
    if poor_sids:
        print(f"\n  ⚠ Poor subjects — consider re-registering with full SyN:")
        for sid in poor_sids:
            print(f"    {sid}")

    # ── Figure 1: CC Distribution Histogram ───────────────────────────────────
    fig1, ax = plt.subplots(figsize=(8, 4))

    # Colour bars by quality tier
    colors = ['#2ecc71' if c > 0.70 else ('#f39c12' if c > 0.50 else '#e74c3c')
              for c in cc_vals]
    ax.bar(range(len(cc_vals)), sorted(cc_vals, reverse=True),
           color=sorted(colors, reverse=True), edgecolor='none', width=0.8)
    ax.axhline(0.70, color='#2ecc71', linewidth=1.5, linestyle='--', label='Good threshold (0.70)')
    ax.axhline(0.50, color='#e74c3c', linewidth=1.5, linestyle='--', label='Poor threshold (0.50)')
    ax.set_xlabel('Subjects (sorted by CC)', fontsize=11)
    ax.set_ylabel('Cross-Correlation (CC)', fontsize=11)
    ax.set_title(
        f'Registration Quality — Session {SESSION_ID}\n'
        f'Mean CC={np.mean(cc_vals):.3f}  |  '
        f'Good: {n_good}  Marginal: {n_marginal}  Poor: {n_poor}',
        fontsize=11
    )
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Annotate CN vs EMCI counts
    n_cn_good   = sum(1 for r in results if r['quality']=='GOOD'  and r['group']=='CN')
    n_emci_good = sum(1 for r in results if r['quality']=='GOOD'  and r['group']=='EMCI')
    ax.text(0.98, 0.05,
            f"CN good: {n_cn_good}  |  EMCI good: {n_emci_good}",
            transform=ax.transAxes, ha='right', fontsize=9, color='gray')

    plt.tight_layout()
    hist_path = REG_ROOT / f'quality_histogram_session_{SESSION_ID}.png'
    plt.savefig(str(hist_path), dpi=120, bbox_inches='tight')
    plt.show()
    print(f"\n  Histogram saved → {hist_path.name}")

    # ── Figure 2: Visual grid — all subjects (axial mid-slice) ────────────────
    # Three panels per subject: MNI template | registered mean fMRI | abs diff
    # Colour-coded border: green=GOOD, orange=MARGINAL, red=POOR
    n      = len(results)
    n_cols = 3          # fixed: MNI | registered | diff
    fig2, axes2 = plt.subplots(n, n_cols, figsize=(n_cols * 4, n * 3.2))
    if n == 1:
        axes2 = axes2[np.newaxis, :]

    quality_colors = {'GOOD': '#2ecc71', 'MARGINAL': '#f39c12', 'POOR': '#e74c3c'}

    for row_idx, r in enumerate(results):
        s    = r['crop']
        midz = s[2] // 2
        mc   = r['mc_norm'][:s[0], :s[1], :s[2]]
        rc   = r['rc_norm'][:s[0], :s[1], :s[2]]
        diff = np.abs(mc[:, :, midz] - rc[:, :, midz])
        qcol = quality_colors[r['quality']]

        axes2[row_idx, 0].imshow(mc[:, :, midz].T, cmap='gray', origin='lower')
        axes2[row_idx, 0].set_title('MNI template', fontsize=7)

        axes2[row_idx, 1].imshow(rc[:, :, midz].T, cmap='gray', origin='lower')
        axes2[row_idx, 1].set_title(
            f"{r['sid']}  ({r['group']})\nCC={r['cc']:.3f}  NMI={r['mi']:.3f}",
            fontsize=7
        )

        axes2[row_idx, 2].imshow(diff.T, cmap='hot', origin='lower', vmin=0, vmax=0.4)
        axes2[row_idx, 2].set_title(f"abs diff  div={r['div']:.3f}", fontsize=7)

        # Colour-coded border around the middle panel for quick visual scan
        for spine in axes2[row_idx, 1].spines.values():
            spine.set_edgecolor(qcol)
            spine.set_linewidth(3)

        for ax in axes2[row_idx]:
            ax.axis('off')

    plt.suptitle(
        f'Full Registration Visual Check — Session {SESSION_ID}  '
        f'({n_good} good / {n_marginal} marginal / {n_poor} poor)',
        fontsize=11, y=1.002
    )
    plt.tight_layout()
    grid_path = REG_ROOT / f'quality_grid_session_{SESSION_ID}.png'
    plt.savefig(str(grid_path), dpi=100, bbox_inches='tight')
    plt.show()
    print(f"  Visual grid saved  → {grid_path.name}")

    # ── Save per-subject metrics to CSV ───────────────────────────────────────
    metrics_df = pd.DataFrame([{
        'original_id': r['sid'],
        'group'      : r['group'],
        'method'     : r['method'],
        'cc'         : round(r['cc'],  4),
        'nmi'        : round(r['mi'],  4),
        'divergence' : round(r['div'], 4),
        'quality'    : r['quality'],
    } for r in results])

    csv_path = REG_ROOT / f'quality_metrics_session_{SESSION_ID}.csv'
    metrics_df.to_csv(str(csv_path), index=False)
    print(f"  Metrics CSV saved  → {csv_path.name}")

    return metrics_df


# Run the full quality check
quality_metrics = full_quality_check(session_df, session_progress)
